In [1]:
import serial
import serial.tools.list_ports
import subprocess, os
import sys
import time


In [3]:
def serial_ports():
    ports = serial.tools.list_ports.comports()
    Flasherport = []
    for port, desc, hwid in sorted(ports):
            if hwid[:4] == "USB ":
                devices = hwid[4:].split(" ")
                if len(devices) > 1 and len(devices[0]) > 9:
                    if devices[0][-9:] == "1A86:55D4":
                        Flasherport.append(port)

    return Flasherport

serial_ports()

['COM34']

In [ ]:
while True:
    flash_new_ambit = False

    with serial.Serial('COM34', 115200) as ser:
        t0 = time.time()
        s = ""
        counter = 0
        while time.time() - t0 < 5:
            while (ser.in_waiting > 0) and (counter < 300):
                c = ser.read()
                if c > bytes([9]) and c < bytes([127]):
                    counter += 1
                    s += c.decode()

                if (counter > 20):
                    if "invalid header" in s:
                        flash_new_ambit = True
                        break
                    else:
                        if c.isascii():
                            print(c.decode(), end="")
            if (counter >= 49): break
            if flash_new_ambit: break
            time.sleep(.25)
            print(".", end="")

In [35]:
def ser_get_lines(ser: serial.Serial, n:int = -1, no_resp_timeout:float = 0.5, max_timeout:float = 50, display:bool = False, ter:str = "") -> list[str]:
    start_t0:float = time.perf_counter()
    last_char_t:float = time.perf_counter()
    a_line:str = ""
    ret_list:list[str] = []
    counter:int = 0
    if n == -1: n = 10000

    while True:
        if time.perf_counter() - start_t0 > max_timeout: break
        if time.perf_counter() - last_char_t > no_resp_timeout: break
        if len(ret_list) >= n: break

        while (ser.in_waiting > 0):
            c:bytes = ser.read()

            if c.isascii():
                _char:str = c.decode()
                if _char == '\n' or _char == '\r':
                    if counter > 0:
                        ret_list += [a_line]
                        if (display): print(a_line)
                        if len(ter) > 0 and (a_line == ter): return ret_list
                        a_line = ""
                    counter = 0

                else:
                    a_line += _char
                    counter += 1
                    last_char_t = time.perf_counter()
            if time.perf_counter() - start_t0 > max_timeout: break
            if counter > 100000: break
        time.sleep(.05)


    if len(a_line)>0:
        ret_list += [a_line]

    return ret_list

In [36]:
with serial.Serial('COM34', 115200) as ser:
    ser.write(b"reboot\r\n")
    ser_get_lines(ser, display=True)

ESP-ROM:esp32c3-api1-20210207
Build:Feb  7 2021
rst:0xc (RTC_SW_CPU_RST),boot:0xd (SPI_FAST_FLASH_BOOT)
Saved PC:0x403819c4
SPIWP:0xee
mode:DIO, clock div:1
load:0x3fcd5820,len:0x1144
load:0x403cc710,len:0xad8
load:0x403ce710,len:0x2f80
entry 0x403cc710
BOOT
256223
ADPD Found, chip version: 2
Metadata: lon:1.000000	lat:1.000000	alt:1.000000	time:0	acc:1.000000	vacc:1.000000	info1:New_Ambit	x:0.000000	y:0.000000	z:0.000000
Calibration: Name:AmbitV002	Actinic:0.100000	Spec:1.000000	Emit:1.000000	Sun:1.000000	Temp_offset:0.000000	Temp_slope:1.000000
Calibration: Act_50:5	Act_100:4	Act_150:3	Act_200:2	Act_250:1
Calibration: ADPD: 0	0	0	0	0	0
MLX: 5832480	78523296	0	6400	4907706	5832480	64774732	-51539608	-35734128	16384	0	10752	10752	72641246	
FW: MAC:f80b44a89110	Size:539216	Date:Mar 26 2025
FW: 0.0.3


In [ ]:
# 0: Conn/NA    1: New NoFW     2: has FW   3: ADPD exist   -2: No ADPD
Ambit_status:int = -1
with serial.Serial('COM34', 115200) as ser:
    t0:int = time.perf_counter()
    loop_contiune:bool = True
    Ambit_status = -1
    
    while time.perf_counter() - t0 < 50:
        if not loop_contiune: break
        _lines:list[str] = ser_get_lines(ser, display=True, ter = "invalid header: 0xffffffff")
        if len(_lines) == 0:
            time.sleep(.05)
            continue

        for _l in _lines:
            if "invalid header: 0xffffffff" in _l:
                print("NEW ambit Found")
                loop_contiune = False
                Ambit_status = 1

            if ("esp" in _l) or ("load" in _l) or ("BOOT" in _l):
                Ambit_status = 2


            if ("ADPD Found" in _l):
                Ambit_status = 3
            else:
                if ("BOOT" in _l) and ("Metadata" in _l):
                    print("ADPD Not Found!")
                    Ambit_status = -2

        

ESP-ROM:esp32c3-api1-20210207
Build:Feb  7 2021
rst:0x1 (POWERON),boot:0xd (SPI_FAST_FLASH_BOOT)
SPIWP:0xee
mode:DIO, clock div:1
load:0x3fcd5820,len:0x1144
load:0x403cc710,len:0xad8
load:0x403ce710,len:0x2f80
entry 0x403cc710
ESP-ROM:esp32c3-api1-20210207
Build:Feb  7 2021
rst:0x1 (POWERON),boot:0xd (SPI_FAST_FLASH_BOOT)
SPIWP:0xee
mode:DIO, clock div:1
load:0x3fcd5820,len:0x1144
load:0x403cc710,len:0xad8
load:0x403ce710,len:0x2f80
entry 0x403cc710
ESP-ROM:esp32c3-api1-20210207
Build:Feb  7 2021
rst:0x1 (POWERON),boot:0xd (SPI_FAST_FLASH_BOOT)
SPIWP:0xee
mode:DIO, clock div:1
load:0x3fcd5820,len:0x1144
load:0x403cc710,len:0xad8
load:0x403ce710,len:0x2f80
entry 0x403cc710
BOOT
256218
ADPD Found, chip version: 2
Metadata: lon:1.000000	lat:1.000000	alt:1.000000	time:0	acc:1.000000	vacc:1.000000	info1:New_Ambit	x:0.000000	y:0.000000	z:0.000000
Calibration: Name:AmbitV002	Actinic:0.100000	Spec:1.000000	Emit:1.000000	Sun:1.000000	Temp_offset:0.000000	Temp_slope:1.000000
Calibration: Act_50: